In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
#Importar os dados e verificar dimens~oes
df_train = pd.read_pickle('Xtrain.pkl')
#print(df_train.shape)
#Concatenar todas as poses e sequ^encias numa matriz de dados.
#Esta matriz vai ser usada na maior parte do projeto
X = np.concatenate(df_train['Skeleton_Sequence'].to_numpy())
#print(X.shape) # N X 66
#Normaliza¸c~ao dos dados - centrar + escalamento (recomendando e cr´ıtico para o PCA)
scaler = StandardScaler()
X_norm = scaler.fit_transform(X)
#Aceder a uma sequ^encia espec´ıfica do dataframe, por exemplo a primeira
#seq = df_train.iloc[0]['Skeleton_Sequence']
#print(seq.shape) # T X 66





# Não MODIFICAR A SEED
np.random.seed(555)

def kminit(X, K, flag):
    #Inicialização - pode ser aleatória ou recorrendo ao k-means++
    # A inicialização do primeiro centróide é fixa
    N = X.shape[0]
    C = []
    primeiro = np.random.randint(N)
    C.append(X[primeiro])


    #complete o código para inicializar os K-1 centróides de acordo com a flag

    if flag == 'random':
        #o codigo já pos um centroide em C, faltam 
        retirarescolhido = np.delete(np.arange(N), primeiro)
        #sorteamos K-1 indices diferentes
        indices = np.random.choice(retirarescolhido, size=K-1, replace=False)
        #juntar a lista de centroides
        for idx in indices:
            C.append(X[idx])

    elif flag == 'km++':
        for _ in range(K-1):
            centroides_escolhidos = np.array(C)
            #distancia ao quadrado de cada ponto a cada centroide (N,m)
            distancia = np.sum((X[:, np.newaxis, :] - centroides_escolhidos[np.newaxis, :, :]) ** 2, axis=2)
            #distancia ao centroide + proximo
            minima_distancia = np.min(distancia, axis=1)
            probabilidade = minima_distancia/np.sum(minima_distancia)

            #sortear ponto
            idx = np.random.choice(N, p=probabilidade)
            C.append(X[idx])

    else:
        print("A flag deve ser random ou km++")

    return np.array(C)




def kmeans_custo(X, C, s):
#Calcula a funcão de custo do k-means
#centroide de cada ponto

    centros_atribuidos = C[s]

    #distancia euclidiana ao quadrado
    distancia_J = np.sum((X - centros_atribuidos)** 2, axis=1)
    #custo total
    J = np.sum(distancia_J)


#devolve a distancia ao quad de cada ponto ao seu centroide e a soma de tudo = J
    return distancia_J, J



#R1 iv - associar pontos aos centróides

def kmeans_atribuir(X,C):
    #Associa cada ponto ao centróide mais próximo.
    #distancia ao quadrado de cada ponto a cada centroide
    dist = np.sum((X[:, np.newaxis, :] - C[np.newaxis, :, :]) ** 2, axis=2)
    #indice do centroide + perto
    return np.argmin(dist, axis=1)


def kmeans(X, K, flag, max_iter=300, tol=1e-6):
#Aplica o algoritm k-means a matriz de dados X.
    N, d = X.shape

    #inicializar
    centroides = kminit(X, K, flag)
    J_inicial = None

    for _ in range(max_iter):
        #atribuição
        atribuir = kmeans_atribuir(X, centroides)

        #atualizar
        centroide_novo = np.zeros((K, d))
        for k in range(K):
            pontos_k = X[atribuir == k]
            if len(pontos_k) > 0:
                centroide_novo[k] = pontos_k.mean(axis=0)
            else:
                #cluster vazio - começa num ponto qualquer
                centroide_novo[k] = X[np.random.randint(N)]
        centroides = centroide_novo

        #paragem = J nao muda
        _, J_atual = kmeans_custo(X, centroides, atribuir)
        if J_inicial is not None and abs(J_inicial - J_atual) < tol: 
            break
        J_inicial = J_atual

    #atribuição final
    atribuir = kmeans_atribuir(X, centroides)
    return centroides, atribuir







In [ ]:
centroides, atribuir = kmeans(X_norm, 5, 'km++')
print(centroides.shape, atribuir.shape, np.bincount(atribuir))
_, J = kmeans_custo(X_norm, centroides, atribuir)
print("J =", J)

In [ ]:

# R2A 


import matplotlib.pyplot as plt

testar_k = range(1, 15)          # testar K de 1 a 15
custos_random = []

for K in testar_k:
    centroides, atribuir = kmeans(X_norm, K, 'random')
    _, J = kmeans_custo(X_norm, centroides, atribuir)
    custos_random.append(J)
    print(f"K = {K:2d}  ->  J = {J:.2f}")

plt.figure(figsize=(8, 5))
plt.plot(testar_k, custos_random, 'o-')
plt.xlabel('Número de centróides K')
plt.ylabel('Função de custo J')
plt.title('R2.a) Custo do k-means em função de K (inicialização aleatória)')
plt.xticks(list(testar_k))
plt.grid(True, alpha=0.3)
plt.savefig('PartI_R2a.jpg', dpi=150, bbox_inches='tight')
plt.show()

# quanto o custo desce de K-1 para K (ajuda a encontrar o cotovelo)
ganhos = -np.diff(custos_random)
ganho_relativo = ganhos / np.array(custos_random[:-1]) * 100
for K, g, r in zip(list(testar_k)[1:], ganhos, ganho_relativo):
    print(f"K = {K-1:2d} -> {K:2d}: J desce {g:9.2f}  ({r:5.1f}%)")

In [ ]:

#R2B



#custo x K, inicialização k-means++ 
custos_kmpp = []

for K in testar_k:
    centroides, atribuir = kmeans(X_norm, K, 'km++')
    _, J = kmeans_custo(X_norm, centroides, atribuir)
    custos_kmpp.append(J)
    print(f"K = {K:2d}  ->  J = {J:.2f}")

plt.figure(figsize=(8, 5))
plt.plot(testar_k, custos_random, 'o-', label='Aleatória')
plt.plot(testar_k, custos_kmpp, 's-', label='k-means++')
plt.xlabel('Número de centróides K')
plt.ylabel('Função de custo J')
plt.title('Comparação das inicializações (1 execução)')
plt.xticks(list(testar_k))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# quanto o custo desce de K-1 para K
ganhos = -np.diff(custos_kmpp)
ganho_relativo = ganhos / np.array(custos_kmpp[:-1]) * 100
for K, g, r in zip(list(testar_k)[1:], ganhos, ganho_relativo):
    print(f"K = {K-1:2d} -> {K:2d}: J desce {g:9.2f}  ({r:5.1f}%)")

In [ ]:
#R2C




#20 execsde cada inicialização 
n_runs = 20
testar_k_lista = list(testar_k)

J_random = np.zeros((n_runs, len(testar_k_lista)))   # linhas = execuções, colunas = K
J_kmpp   = np.zeros((n_runs, len(testar_k_lista)))

for r in range(n_runs):
    for i, K in enumerate(testar_k_lista):
        centroides, atribuir = kmeans(X_norm, K, 'random')
        _, J_random[r, i] = kmeans_custo(X_norm, centroides, atribuir)

        centroides, atribuir = kmeans(X_norm, K, 'km++')
        _, J_kmpp[r, i] = kmeans_custo(X_norm, centroides, atribuir)
    print(f"execução {r+1}/{n_runs} concluída")

# média e desvio padrão para cada K (ao longo das 20 execuções)
media_rand, dp_rand = J_random.mean(axis=0), J_random.std(axis=0)
media_kpp,  dp_kpp  = J_kmpp.mean(axis=0),   J_kmpp.std(axis=0)

plt.figure(figsize=(9, 5))
plt.errorbar(testar_k_lista, media_rand, yerr=dp_rand, fmt='o-', capsize=4, label='Aleatória')
plt.errorbar(testar_k_lista, media_kpp,  yerr=dp_kpp,  fmt='s-', capsize=4, label='k-means++')
plt.xlabel('Número de centróides K')
plt.ylabel('Função de custo J (média ± desvio padrão)')
plt.title('R2.c) Custo do k-means em 20 execuções')
plt.xticks(testar_k_lista)
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('PartI_R2c.jpg', dpi=150, bbox_inches='tight')
plt.show()

print(" K |  média aleat. |  dp aleat. |  média km++ |   dp km++")
for i, K in enumerate(testar_k_lista):
    print(f"{K:2d} | {media_rand[i]:13.1f} | {dp_rand[i]:10.1f} | {media_kpp[i]:11.1f} | {dp_kpp[i]:9.1f}")

In [ ]:
#Auxiliar R2C


diferenca_media = (media_kpp - media_rand) / media_rand * 100   # negativo = km++ melhor
print(" K | dif. média (%) | dp aleat. | dp km++ | menor dp")
for i, K in enumerate(testar_k_lista):
    menor = 'km++' if dp_kpp[i] < dp_rand[i] else 'aleat.'
    print(f"{K:2d} | {diferenca_media[i]:+13.1f}  | {dp_rand[i]:9.0f} | {dp_kpp[i]:7.0f} | {menor}")

print("\nK em que km++ tem média menor:", [K for K, d in zip(testar_k_lista, diferenca_media) if d < 0])
print("K em que km++ tem dp menor:    ", [K for K, a, b in zip(testar_k_lista, dp_rand, dp_kpp) if b < a])

In [ ]:
#R3A


# visualização das poses dos centróides 
LIGACOES = [
    # cara
    (0, 1), (1, 2), (2, 3), (3, 7), (0, 4), (4, 5), (5, 6), (6, 8), (9, 10),
    # tronco
    (11, 12), (11, 23), (12, 24), (23, 24),
    # braço (11) e mão
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19),
    # braço (12) e mão
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20),
    # perna (23) e pé
    (23, 25), (25, 27), (27, 29), (29, 31), (27, 31),
    # perna (24) e pé
    (24, 26), (26, 28), (28, 30), (30, 32), (28, 32),
]

def desenhar_pose(pose, ax, titulo=None):
    # pose = [u0, v0, u1, v1, ..., u32, v32]
    u = pose[0::2]     #par -> coordenadas u
    v = pose[1::2]     #ímpar -> coordenadas v
    for a, b in LIGACOES:
        ax.plot([u[a], u[b]], [v[a], v[b]], 'k-', lw=1.5)
    ax.scatter(u, v, c='red', s=12, zorder=3)
    ax.set_aspect('equal')
    ax.invert_yaxis()            # em imagem, v cresce para baixo
    ax.set_xlabel('u')
    ax.set_ylabel('v')
    if titulo:
        ax.set_title(titulo)


# centróides do modelo escolhido 
K_escolhido = 11
init_escolhida = 'km++'

centroides, atribuir = kmeans(X_norm, K_escolhido, init_escolhida)
centroides_reais = scaler.inverse_transform(centroides)   # desfazer a normalização
tamanhos = np.bincount(atribuir, minlength=K_escolhido)

n_cols = 4
n_rows = int(np.ceil(K_escolhido / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4.5 * n_rows), sharex=True, sharey=True)
axes = axes.ravel()

for k in range(K_escolhido):
    desenhar_pose(centroides_reais[k], axes[k], f'Centróide {k} ({tamanhos[k]} poses)')
for ax in axes[K_escolhido:]:
    ax.axis('off')       # esconder os quadrados vazios

plt.tight_layout()
plt.savefig('PartI_R3a.jpg', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== PARTE II – R5 (código do David, adaptado: usa o X, o X_norm e o scaler já definidos) =====
import os
os.makedirs("figures", exist_ok=True)
from sklearn.decomposition import PCA

# --- R5.a ---
# PCA sem definir n_components nem threshold
pca = PCA()
pca.fit(X_norm)

# Variância explicada por cada componente e variância acumulada
var_exp = pca.explained_variance_ratio_          # shape (66,)
var_cum = np.cumsum(var_exp)                     # shape (66,)

# Primeiro número de componentes cuja variância acumulada chega a 95%
n_95 = int(np.argmax(var_cum >= 0.95)) + 1
print("Componentes para 95% da variância:", n_95)
print("Variância acumulada com esse número:", f"{var_cum[n_95 - 1]:.4%}")
print(f"Número total de componentes: {pca.n_components_}")
print(f"Variância explicada pelas primeiras {n_95 - 1}: {var_cum[n_95 - 2]:.4%}")
print(f"Variância explicada pelas primeiras {n_95}: "
      f"{var_cum[n_95 - 1]:.4%}")

# Figura de suporte
plt.figure()
plt.plot(range(1, len(var_cum) + 1), var_cum, marker="o", markersize=3)
plt.axhline(0.95, linestyle="--", color="red", label="95%")
plt.axvline(n_95, linestyle="--", color="gray", label=f"{n_95} componentes")
plt.xlabel("Número de componentes principais")
plt.ylabel("Variância explicada acumulada")
plt.title("PCA: variância explicada acumulada")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("figures/PartII5a.jpg", dpi=300)

# --- R5.b ---

# Nomes das 66 caracteristicas originais
features = []
for j in range(33):
    features.append(f"Marcador {j} - u")
    features.append(f"Marcador {j} - v")

# Analisar as 8 componentes que explicam pelo menos 95%
for i in range(n_95):
    pesos = pca.components_[i]

    # Ordenar por valor absoluto do peso, do maior para o menor
    indices = np.argsort(np.abs(pesos))[::-1][:10]

    print(f"\nComponente principal {i + 1}")
    print(f"Variancia explicada: "
          f"{pca.explained_variance_ratio_[i]:.4%}")
    print("Top 10 caracteristicas:")

    for idx in indices:
        print(f"{features[idx]}: peso = {pesos[idx]:.4f}")

# Figura de apoio: pesos de cada feature nas 2 primeiras componentes
fig, axs = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for c, ax in enumerate(axs):
    ax.bar(range(66), pca.components_[c])
    ax.set_ylabel(f"Peso em PC{c + 1}")
    ax.grid(True, axis="y")
axs[-1].set_xticks(range(66))
axs[-1].set_xticklabels(
    [f"{i // 2}{'u' if i % 2 == 0 else 'v'}" for i in range(66)],
    rotation=90, fontsize=6
)
axs[-1].set_xlabel("Característica original (marcador + coordenada)")
plt.tight_layout()
plt.savefig("figures/PartII5b.jpg", dpi=300)
plt.show()


In [ ]:
# ===== PARTE II – R6 (código do David) =====
# --- R6: PCA implementado de raiz (apenas numpy) ---

def pca_fit(X):
    """
    Ajusta o PCA a X (N, d).
    Devolve: media (d,), valores_proprios (d,), W (d, d)
    onde as colunas de W são as componentes principais, ordenadas
    da que explica mais variância para a que explica menos.
    """
    media = X.mean(axis=0)
    Xc = X - media
    cov = (Xc.T @ Xc) / (X.shape[0] - 1)        # matriz de covariância (d, d)

    valores, vetores = np.linalg.eigh(cov)       # eigh: para matrizes simétricas
    ordem = np.argsort(valores)[::-1]            # eigh devolve ordem crescente
    return media, valores[ordem], vetores[:, ordem]


def pca_project(X, media, W, r):
    """Projeta X (N, d) nas primeiras r componentes -> Z (N, r)."""
    return (X - media) @ W[:, :r]


def pca_reconstruct(Z, media, W):
    """Reconstrói X_hat (N, d) a partir de Z (N, r)."""
    r = Z.shape[1]
    return Z @ W[:, :r].T + media

# --- R6.a ---
media, valores, W = pca_fit(X_norm)
var_ratio = valores / valores.sum()

Z2 = pca_project(X_norm, media, W, r=2)         # (N, 2)
print("Shape de Z2:", Z2.shape)
print("Variância explicada pelas 2 componentes:", var_ratio[:2].sum())

plt.figure(figsize=(7, 6))
plt.scatter(Z2[:, 0], Z2[:, 1], s=5, alpha=0.4)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Dados projetados em 2 componentes principais")
plt.grid(True)
plt.tight_layout()
plt.savefig("figures/PartII6a.jpg", dpi=300)

# --- R6.b ---
# Reconstrução com r = 2
Z2 = pca_project(X_norm, media, W, r=2)
X_norm_rec = pca_reconstruct(Z2, media, W)          # espaço normalizado
X_rec = scaler.inverse_transform(X_norm_rec)        # espaço original

# MSE global (média sobre todas as poses e todas as características)
mse_orig = np.mean((X - X_rec) ** 2)
mse_norm = np.mean((X_norm - X_norm_rec) ** 2)
print("MSE (espaço original):", mse_orig)
print("MSE (espaço normalizado):", mse_norm)
print("Verificação: 1 - variância explicada =", 1 - var_ratio[:2].sum())

# Erro por característica (66,) e por marcador (33,), somando u e v
erro_feat = np.mean((X - X_rec) ** 2, axis=0)
erro_marc = erro_feat[0::2] + erro_feat[1::2]

# Marcadores com maior erro
ordem = np.argsort(erro_marc)[::-1]
print("\nMarcadores com maior erro de reconstrução:")
for j in ordem[:10]:
    print(f"Marcador {j}: {erro_marc[j]:.6f}")

# Figura
plt.figure(figsize=(10, 4))
plt.bar(range(33), erro_marc)
plt.xlabel("Marcador")
plt.ylabel("Erro quadrático médio (u² + v²)")
plt.title("Erro de reconstrução por marcador (r = 2)")
plt.xticks(range(33))
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig("figures/PartII6b.jpg", dpi=300)

# --- R6.c ---
def avaliar_reconstrucao(r):
    """Projeta, reconstrói e mede o erro para r componentes."""
    Z = pca_project(X_norm, media, W, r)
    X_norm_rec = pca_reconstruct(Z, media, W)
    X_rec = scaler.inverse_transform(X_norm_rec)

    var_expl = var_ratio[:r].sum()
    mse_orig = np.mean((X - X_rec) ** 2)
    mse_norm = np.mean((X_norm - X_norm_rec) ** 2)

    erro_feat = np.mean((X - X_rec) ** 2, axis=0)
    erro_marc = erro_feat[0::2] + erro_feat[1::2]
    return Z, var_expl, mse_orig, mse_norm, erro_marc

r_95 = n_95   # 8, obtido na R5.a
Z8, var8, mse_orig8, mse_norm8, erro_marc8 = avaliar_reconstrucao(r_95)
_, var2, mse_orig2, mse_norm2, erro_marc2 = avaliar_reconstrucao(2)

print("Shape de Z (r = 8):", Z8.shape)
print(f"\n{'':22s}{'r = 2':>12s}{'r = ' + str(r_95):>12s}")
print(f"{'Variância explicada':22s}{var2:12.4%}{var8:12.4%}")
print(f"{'MSE (original)':22s}{mse_orig2:12.6f}{mse_orig8:12.6f}")
print(f"{'MSE (normalizado)':22s}{mse_norm2:12.6f}{mse_norm8:12.6f}")
print("Verificação: 1 - variância explicada (r = 8) =", 1 - var8)

ordem8 = np.argsort(erro_marc8)[::-1]
print("\nMarcadores com maior erro (r = 8):")
for j in ordem8[:10]:
    print(f"Marcador {j}: {erro_marc8[j]:.6f}   (r = 2: {erro_marc2[j]:.6f})")

# Figura: erro por marcador, r = 2 vs r = 8, com o mesmo eixo
x = np.arange(33)
plt.figure(figsize=(10, 4))
plt.bar(x - 0.2, erro_marc2, width=0.4, label="r = 2")
plt.bar(x + 0.2, erro_marc8, width=0.4, label=f"r = {r_95}")
plt.xlabel("Marcador")
plt.ylabel("Erro quadrático médio (u² + v²)")
plt.title("Erro de reconstrução por marcador")
plt.xticks(range(33))
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig("figures/PartII6c.jpg", dpi=300)
plt.show()


In [ ]:
#R4A


# utlizando funções do PCA da R6, do pca.py
def pca_fit(X):
    media = X.mean(axis=0)
    Xc = X - media
    cov = (Xc.T @ Xc) / (X.shape[0] - 1)        # matriz de covariância (d, d)
    valores, vetores = np.linalg.eigh(cov)
    ordem = np.argsort(valores)[::-1]            # da maior variância para a menor
    return media, valores[ordem], vetores[:, ordem]

def pca_project(X, media, W, r):
    return (X - media) @ W[:, :r]


# ===== R4.a: projeção 2D dos dados e dos centróides =====
assert centroides.shape == (11, 66), "centroides não é o modelo da R3a"

media, valores, W = pca_fit(X_norm)
var_ratio = valores / valores.sum()

Z2 = pca_project(X_norm, media, W, r=2)          # poses em 2D, (8374, 2)
C2 = pca_project(centroides, media, W, r=2)      # centróides em 2D, (11, 2)

cores = plt.cm.tab20(np.linspace(0, 1, K_escolhido))

plt.figure(figsize=(9, 7))
for k in range(K_escolhido):
    pts = Z2[atribuir == k]
    plt.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.35, color=cores[k],
                label=f'Cluster {k} ({len(pts)})')

# centróides: marcador diferente, com o número ao lado
plt.scatter(C2[:, 0], C2[:, 1], marker='X', s=220, c='black',
            edgecolors='white', linewidths=1.5, zorder=3, label='Centróides')
for k, (x, y) in enumerate(C2):
    plt.annotate(str(k), (x, y), xytext=(6, 6), textcoords='offset points',
                 fontsize=11, fontweight='bold')

plt.xlabel(f'Componente principal 1 ({var_ratio[0]:.1%} da variância)')
plt.ylabel(f'Componente principal 2 ({var_ratio[1]:.1%} da variância)')
plt.title('R4.a) Projeção 2D (PCA) dos dados e dos centróides do k-means (K = 11)')
leg = plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
for h in leg.legend_handles[:-1]:      # clusters: pontos maiores e opacos
    h.set_sizes([40])
    h.set_alpha(1)
leg.legend_handles[-1].set_sizes([90]) # centróides: X de tamanho normal
plt.grid(True, alpha=0.3)
plt.savefig('PartI4a.jpg', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#R7A



from sklearn.mixture import GaussianMixture

var_acum = np.cumsum(var_ratio)
r_95 = int(np.argmax(var_acum >= 0.95)) + 1
Z = pca_project(X_norm, media, W, r=r_95)
print(f"r = {r_95} | variância explicada = {var_acum[r_95-1]:.2%} | Z: {Z.shape}")


# ===== R7.a: escolha do número de componentes do GMM =====
Ks_gmm = list(range(1, 31))
bic, aic = [], []

for k in Ks_gmm:
    gmm = GaussianMixture(n_components=k, covariance_type='full',
                          n_init=5, random_state=555)
    gmm.fit(Z)
    bic.append(gmm.bic(Z))
    aic.append(gmm.aic(Z))
    print(f"K = {k:2d} | BIC = {bic[-1]:10.1f} | AIC = {aic[-1]:10.1f}")

k_bic = Ks_gmm[int(np.argmin(bic))]
k_aic = Ks_gmm[int(np.argmin(aic))]
print(f"\nMínimo do BIC: K = {k_bic} | Mínimo do AIC: K = {k_aic}")

plt.figure(figsize=(8, 5))
plt.plot(Ks_gmm, bic, 'o-', label='BIC')
plt.plot(Ks_gmm, aic, 's-', label='AIC')
plt.axvline(k_bic, color='tab:blue', linestyle='--', alpha=0.6)
plt.axvline(k_aic, color='tab:orange', linestyle='--', alpha=0.6)
plt.xlabel('Número de componentes do GMM')
plt.ylabel('Critério de informação (menor = melhor)')
plt.title(f'R7.a) BIC e AIC do GMM (covariâncias completas, r = {r_95})')
plt.xticks(Ks_gmm)
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('PartIII7a.jpg', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Auxiliar R7A


desc_bic = -np.diff(bic) / np.array(bic[:-1]) * 100
for k, d in zip(Ks_gmm[1:], desc_bic):
    print(f"K = {k-1:2d} -> {k:2d}: BIC desce {d:5.1f}%")

In [ ]:
# R7B: GMM final e visualização com covariâncias 
from matplotlib.patches import Ellipse

K_gmm = 9   # <- escolhido na R7a
gmm = GaussianMixture(n_components=K_gmm, covariance_type='full',
                      n_init=5, random_state=555).fit(Z)
labels_gmm = gmm.predict(Z)                    # componente mais provável de cada pose
prob_max = gmm.predict_proba(Z).max(axis=1)    # confiança dessa atribuição

# as 2 primeiras colunas de Z são a mesma projeção 2D da R4a
assert np.allclose(Z[:, :2], Z2)

def desenhar_elipse(media2, cov2, ax, cor, n_std):
    """Elipse de n_std desvios padrão de uma gaussiana 2D."""
    vals, vecs = np.linalg.eigh(cov2)              # eixos principais da elipse
    ordem = vals.argsort()[::-1]
    vals, vecs = vals[ordem], vecs[:, ordem]
    angulo = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    largura, altura = 2 * n_std * np.sqrt(vals)
    ax.add_patch(Ellipse(media2, largura, altura, angle=angulo, fill=False,
                         edgecolor=cor, lw=2 if n_std == 1 else 1,
                         linestyle='-' if n_std == 1 else '--'))

cores_gmm = plt.cm.tab10(np.arange(K_gmm))
fig, ax = plt.subplots(figsize=(9, 7))

for k in range(K_gmm):
    pts = Z[labels_gmm == k, :2]
    ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.35, color=cores_gmm[k],
               label=f'Componente {k} ({len(pts)})')
    media2 = gmm.means_[k, :2]                     # média nas 2 primeiras PCs
    cov2 = gmm.covariances_[k][:2, :2]             # bloco 2x2 da covariância
    desenhar_elipse(media2, cov2, ax, cores_gmm[k], n_std=1)
    desenhar_elipse(media2, cov2, ax, cores_gmm[k], n_std=2)

ax.scatter(gmm.means_[:, 0], gmm.means_[:, 1], marker='X', s=200, c='black',
           edgecolors='white', linewidths=1.5, zorder=3, label='Médias')
for k, (x, y) in enumerate(gmm.means_[:, :2]):
    ax.annotate(str(k), (x, y), xytext=(6, 6), textcoords='offset points',
                fontsize=11, fontweight='bold')

ax.set_xlabel(f'Componente principal 1 ({var_ratio[0]:.1%} da variância)')
ax.set_ylabel(f'Componente principal 2 ({var_ratio[1]:.1%} da variância)')
ax.set_title(f'R7.b) GMM (K = {K_gmm}) projetado em 2D, com elipses de 1σ (—) e 2σ (--)')
leg = ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
for h in leg.legend_handles[:-1]:
    h.set_sizes([40]); h.set_alpha(1)
leg.legend_handles[-1].set_sizes([90])
ax.grid(True, alpha=0.3)
plt.savefig('PartIII7b.jpg', dpi=150, bbox_inches='tight')
plt.show()

# tabela de apoio
print(" k | peso  | n poses | confiança média")
for k in range(K_gmm):
    m = labels_gmm == k
    print(f"{k:2d} | {gmm.weights_[k]:.3f} | {m.sum():7d} | {prob_max[m].mean():.3f}")

In [ ]:
#Auxiliar R7B

# quantas poses de cada cluster do k-means (linhas) caem em cada componente do GMM (colunas)
tab = pd.crosstab(pd.Series(atribuir, name='k-means'), pd.Series(labels_gmm, name='GMM'))
print(tab)

In [ ]:
# R7c: poses do GMM no espaço original






def gmm_para_espaco_original(gmm, W, media, scaler):
    #Converte médias e covariâncias do GMM (espaço PCA, r dims) para o espaço
    #original dos esqueletos (66 dims, coordenadas u, v).
    r = gmm.means_.shape[1]
    Wr = W[:, :r]                                   # (66, r)
    D = np.diag(scaler.scale_)                      # desvios padrão do StandardScaler
    poses, covs = [], []
    for k in range(gmm.n_components):
        mu_norm = Wr @ gmm.means_[k] + media                    # (66,)
        cov_norm = Wr @ gmm.covariances_[k] @ Wr.T              # (66, 66)
        poses.append(scaler.inverse_transform(mu_norm[None, :])[0])
        covs.append(D @ cov_norm @ D)
    return np.array(poses), np.array(covs)


def desenhar_pose_cov(pose, cov, ax, cor='tab:blue', n_std=2, titulo=None, pose_ref=None):
    #Desenha uma pose com a elipse de covariância (n_std) de cada marcador.
    #pose_ref: pose do k-means a sobrepor (cinzento), para comparação.
    if pose_ref is not None:
        u_r, v_r = pose_ref[0::2], pose_ref[1::2]
        for a, b in LIGACOES:
            ax.plot([u_r[a], u_r[b]], [v_r[a], v_r[b]], color='0.7', lw=1.5, ls='--')
    u, v = pose[0::2], pose[1::2]
    for j in range(33):
        c2 = cov[2*j:2*j+2, 2*j:2*j+2]              # bloco 2x2 (u_j, v_j)
        vals, vecs = np.linalg.eigh(c2)
        ang = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))   # eixo maior
        ax.add_patch(Ellipse((u[j], v[j]),
                             2 * n_std * np.sqrt(max(vals[1], 0)),
                             2 * n_std * np.sqrt(max(vals[0], 0)),
                             angle=ang, facecolor=cor, alpha=0.15, edgecolor=cor, lw=0.8))
    for a, b in LIGACOES:
        ax.plot([u[a], u[b]], [v[a], v[b]], 'k-', lw=1.5)
    ax.scatter(u, v, c='red', s=10, zorder=3)
    ax.set_aspect('equal')
    if titulo:
        ax.set_title(titulo, fontsize=10)


poses_gmm, covs_gmm = gmm_para_espaco_original(gmm, W, media, scaler)

# cluster do k-means com mais poses em cada componente do GMM (da tabela cruzada)
mapa_km = tab.idxmax(axis=0)

fig, axes = plt.subplots(3, 3, figsize=(13, 15))
for k, ax in enumerate(axes.ravel()):
    km = int(mapa_km[k])
    desenhar_pose_cov(poses_gmm[k], covs_gmm[k], ax, cor=cores_gmm[k],
                      titulo=f'GMM {k} (peso {gmm.weights_[k]:.2f}) | k-means {km}',
                      pose_ref=centroides_reais[km])

# mesma escala em todos os subgráficos e v  cresce p baixo
todas = np.vstack([poses_gmm, centroides_reais])
umin, umax = todas[:, 0::2].min(), todas[:, 0::2].max()
vmin, vmax = todas[:, 1::2].min(), todas[:, 1::2].max()
for ax in axes.ravel():
    ax.set_xlim(umin - 0.15, umax + 0.15)
    ax.set_ylim(vmax + 0.15, vmin - 0.15)        # invertido: cabeça em cima
plt.suptitle('R7.c) Poses médias do GMM (preto) com elipses de 2σ por marcador; '
             'centróide do k-means correspondente a tracejado', fontsize=11)
plt.tight_layout()
plt.savefig('PartIII7c.jpg', dpi=150, bbox_inches='tight')
plt.show()


# Variância por marcador e pose
dp_marcador = np.array([[np.sqrt(np.trace(covs_gmm[k][2*j:2*j+2, 2*j:2*j+2]))
                         for j in range(33)] for k in range(K_gmm)])   # (K, 33)

plt.figure(figsize=(12, 4))
plt.imshow(dp_marcador, aspect='auto', cmap='viridis')
plt.colorbar(label='Desvio padrão total do marcador (u, v)')
plt.xlabel('Marcador'); plt.ylabel('Componente do GMM')
plt.xticks(range(33)); plt.yticks(range(K_gmm))
plt.title('R7.c) Variabilidade de cada marcador em cada pose do GMM')
plt.savefig('PartIII7c_var.jpg', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Q9





from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from sklearn.metrics import silhouette_score

seqs = df_train['Skeleton_Sequence'].to_numpy()
n_seq = len(seqs)
tam = np.array([s.shape[0] for s in seqs])            # nº de frames de cada sequência
lim = np.concatenate([[0], np.cumsum(tam)])           # onde começa cada sequência em X
assert lim[-1] == X.shape[0]

# Descritor 1: assinatura de movimento (quanto varia cada coordenada na sequência)
F_mov = np.array([s.std(axis=0) for s in seqs])                       # (71, 66)
F_mov = F_mov / np.linalg.norm(F_mov, axis=1, keepdims=True)          # padrão, não amplitude

# Descritor 2: saco de poses (fração de frames em cada componente do GMM)
F_pose = np.array([np.bincount(labels_gmm[lim[i]:lim[i+1]], minlength=K_gmm) / tam[i]
                   for i in range(n_seq)])                            # (71, K_gmm)

descritores = {'movimento': F_mov, 'poses': F_pose}
ks = list(range(2, 11))
ligacoes = {}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for lin, (nome, F) in enumerate(descritores.items()):
    Lk = linkage(F, method='ward')
    ligacoes[nome] = Lk
    dendrogram(Lk, ax=axes[lin, 0], no_labels=True)
    axes[lin, 0].set_title(f'Dendrograma (Ward) – descritor de {nome}')
    axes[lin, 0].set_xlabel('Sequências (71)')
    axes[lin, 0].set_ylabel('Distância de fusão')

    sil = [silhouette_score(F, fcluster(Lk, k, 'maxclust')) for k in ks]
    axes[lin, 1].plot(ks, sil, 'o-')
    axes[lin, 1].set_title(f'Silhueta vs nº de grupos – descritor de {nome}')
    axes[lin, 1].set_xlabel('Número de grupos')
    axes[lin, 1].set_ylabel('Silhueta média')
    axes[lin, 1].set_xticks(ks)
    axes[lin, 1].grid(True, alpha=0.3)
    print(nome, '| silhueta:', ' '.join(f'{k}:{s:.2f}' for k, s in zip(ks, sil)))

plt.tight_layout()
plt.savefig('PartIV9a.jpg', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Q9: descrição de cada grupo
n_ex = 3 # o número que escolhemos foi o 3

grupos_mov  = fcluster(ligacoes['movimento'], n_ex, 'maxclust') - 1
grupos_pose = fcluster(ligacoes['poses'],     n_ex, 'maxclust') - 1
print("Concordância entre os dois descritores:")
print(pd.crosstab(pd.Series(grupos_mov, name='movimento'), pd.Series(grupos_pose, name='poses')))

nomes = ['nariz','olho E int','olho E','olho E ext','olho D int','olho D','olho D ext',
         'orelha E','orelha D','boca E','boca D','ombro E','ombro D','cotovelo E',
         'cotovelo D','pulso E','pulso D','mindinho E','mindinho D','indicador E',
         'indicador D','polegar E','polegar D','anca E','anca D','joelho E','joelho D',
         'tornozelo E','tornozelo D','calcanhar E','calcanhar D','pé E','pé D']

prop_ex = np.zeros((n_ex, K_gmm))
for e in range(n_ex):
    m = grupos_mov == e
    energia = F_mov[m].mean(0) ** 2
    energia = energia[0::2] + energia[1::2]          # u + v, por marcador
    top_m = np.argsort(energia)[::-1][:5]
    prop_ex[e] = F_pose[m].mean(0)
    top_p = np.argsort(prop_ex[e])[::-1][:3]
    print(f"\nExercício {e}: {m.sum()} sequências | duração média {tam[m].mean():.0f} frames")
    print("  marcadores que mais se mexem:", ', '.join(nomes[j] for j in top_m))
    print("  componentes GMM mais frequentes:",
          ', '.join(f'GMM {c} ({prop_ex[e, c]:.0%})' for c in top_p))

plt.figure(figsize=(9, 4))
plt.imshow(prop_ex, aspect='auto', cmap='viridis')
plt.colorbar(label='Fração média de frames')
plt.xlabel('Componente do GMM (pose, ver R7c)')
plt.ylabel('Exercício (grupo de sequências)')
plt.xticks(range(K_gmm)); plt.yticks(range(n_ex))
plt.title('Q9) Poses que compõem cada exercício')
plt.savefig('PartIV9b.jpg', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#Auxiliar Q9

print(np.round(F_pose[grupos_mov == 0][:, [5, 6]], 2))


In [ ]:
# ===== PARTE IV – Q10 (código do David, adaptado ao notebook) =====
# Alterações face ao Q10.py: (1) usa as funções kmeans/kmeans_custo do notebook em vez de importar o kmeans.py;
# (2) variáveis renomeadas para não colidirem com o resto do notebook: Z -> Z_med, F -> F_corpo,
#     todas -> todas_q10, sil -> sil_q10.
# A seed é reposta com o MESMO valor do enunciado (555), para os resultados do k-means coincidirem
# com os do Q10.py corrido sozinho (onde a seed era aplicada logo antes desta análise).
np.random.seed(555)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster


os.makedirs("figures", exist_ok=True)

# -------------------------------------------------------------------
# Dados
# -------------------------------------------------------------------
df = pd.read_pickle("Xtrain.pkl")
seqs = [np.asarray(s) for s in df["Skeleton_Sequence"]]

comprimentos = np.array([s.shape[0] for s in seqs])
print("Colunas:", df.columns.tolist())
print("Número de sequências:", len(seqs))
print("min / mediana / max:", comprimentos.min(), np.median(comprimentos), comprimentos.max())
print("Total de poses:", comprimentos.sum())          # deve dar 8374

todas_q10 = np.concatenate(seqs).reshape(-1, 33, 2)
print("u: min/max =", todas_q10[:, :, 0].min().round(3), todas_q10[:, :, 0].max().round(3))
print("v: min/max =", todas_q10[:, :, 1].min().round(3), todas_q10[:, :, 1].max().round(3))


# -------------------------------------------------------------------
# Descritores de forma do corpo
# -------------------------------------------------------------------
PARES = [(11, 12), (23, 24), (11, 13), (13, 15), (12, 14),
         (14, 16), (23, 25), (25, 27), (24, 26), (26, 28)]


def tronco_seq(seq, ar=1.0):
    """Comprimento do tronco (ponto médio dos ombros -> ponto médio das ancas), (T,)."""
    P = seq.reshape(len(seq), 33, 2).astype(float).copy()
    P[:, :, 0] *= ar
    return np.linalg.norm(
        (P[:, 11] + P[:, 12]) / 2 - (P[:, 23] + P[:, 24]) / 2, axis=1
    )


def segmentos_brutos(seq, ar=1.0):
    """Comprimento de cada segmento do corpo, sem normalizar: (T, 10)."""
    P = seq.reshape(len(seq), 33, 2).astype(float).copy()
    P[:, :, 0] *= ar
    return np.stack(
        [np.linalg.norm(P[:, a] - P[:, b], axis=1) for a, b in PARES], axis=1
    )


def segmentos_corpo(seq, ar=1.0):
    """Segmentos normalizados pelo tronco instantâneo: (T, 10)."""
    return segmentos_brutos(seq, ar) / tronco_seq(seq, ar)[:, None]


def descritor_mediana(seq, ar=1.0):
    return np.median(segmentos_corpo(seq, ar), axis=0)


def descritor_q(seq, ar=1.0, q=90):
    # numa projeção 2D os segmentos só encurtam, por isso um percentil
    # alto aproxima melhor o comprimento real do que a mediana
    return np.percentile(segmentos_corpo(seq, ar), q, axis=0)


def descritor_robusto(seq, ar=1.0, q=90):
    # normaliza pelo percentil do tronco da própria sequência (momentos em
    # que o paciente está mais direito), em vez de pelo tronco instantâneo
    tr = np.percentile(tronco_seq(seq, ar), q)
    return np.percentile(segmentos_brutos(seq, ar), q, axis=0) / tr


# -------------------------------------------------------------------
# Variação entre sequências vs. dentro de cada sequência
# -------------------------------------------------------------------
F_corpo = np.array([descritor_mediana(s) for s in seqs])                 # (71, 10)
F_std_dentro = np.array([segmentos_corpo(s).std(axis=0) for s in seqs])

print("\nVariação ENTRE sequências (desvio padrão):", F_corpo.std(axis=0).round(3))
print("Variação DENTRO de uma sequência (média):", F_std_dentro.mean(axis=0).round(3))


# -------------------------------------------------------------------
# Teste de fiabilidade (split-half)
# -------------------------------------------------------------------
def fiabilidade_metades(seqs, f, nome):
    """
    Compara a 1.ª metade de cada sequência com a 2.ª metade de todas as
    sequências. Se o descritor identifica o paciente, a metade da própria
    sequência deve ficar entre as mais próximas.
    """
    A = np.array([f(s[:len(s) // 2]) for s in seqs])
    B = np.array([f(s[len(s) // 2:]) for s in seqs])
    sc = StandardScaler().fit(np.vstack([A, B]))
    A, B = sc.transform(A), sc.transform(B)

    d = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)      # (71, 71)
    propria = np.diag(d)
    posicao = (d < propria[:, None]).sum(axis=1)   # 0 = a própria metade é a mais próxima

    print(f"\n[{nome}]")
    print("  Mediana da posição:", np.median(posicao), f"(ao acaso seria ~{len(seqs) // 2})")
    print("  Fração em que a própria metade é a mais próxima:", round(np.mean(posicao == 0), 3))


fiabilidade_metades(seqs, descritor_mediana, "mediana, ar = 1")
fiabilidade_metades(seqs, descritor_q, "percentil 90, ar = 1")
fiabilidade_metades(seqs, descritor_robusto, "robusto (tronco p90), ar = 1")

AR = 16 / 9
fiabilidade_metades(seqs, lambda s: descritor_mediana(s, AR), f"mediana, ar = {AR:.2f}")
fiabilidade_metades(seqs, lambda s: descritor_q(s, AR), f"percentil 90, ar = {AR:.2f}")
fiabilidade_metades(seqs, lambda s: descritor_robusto(s, AR), f"robusto, ar = {AR:.2f}")


# -------------------------------------------------------------------
# Matrizes de descritores normalizadas
# -------------------------------------------------------------------
F_norm = StandardScaler().fit_transform(F_corpo)
F90 = np.array([descritor_q(s) for s in seqs])
F90_norm = StandardScaler().fit_transform(F90)
F_rob = np.array([descritor_robusto(s) for s in seqs])
F_rob_norm = StandardScaler().fit_transform(F_rob)


def sil_q10(Fn, z):
    return silhouette_score(Fn, z) if len(np.unique(z)) > 1 else float("nan")


# -------------------------------------------------------------------
# Silhouette por número de grupos (k-means do grupo, km++)
# -------------------------------------------------------------------
print("\nSilhouette por número de grupos:")
print("   K | mediana |   p90   | robusto")
for K in range(2, 16):
    _, z1 = kmeans(F_norm, K, "km++")
    _, z2 = kmeans(F90_norm, K, "km++")
    _, z3 = kmeans(F_rob_norm, K, "km++")
    print(f"  {K:2d} | {sil_q10(F_norm, z1):7.3f} | {sil_q10(F90_norm, z2):7.3f} | {sil_q10(F_rob_norm, z3):7.3f}")


# -------------------------------------------------------------------
# Dendrogramas (mediana e robusto)
# -------------------------------------------------------------------
Z_med = linkage(F_norm, method="ward")
plt.figure(figsize=(12, 4))
dendrogram(Z_med, color_threshold=0)
plt.xlabel("Sequência")
plt.ylabel("Distância (Ward)")
plt.title("Dendrograma das sequências (descritor: mediana)")
plt.tight_layout()
plt.savefig("figures/PartIV10.jpg", dpi=300)

Z_rob = linkage(F_rob_norm, method="ward")
plt.figure(figsize=(12, 4))
dendrogram(Z_rob, color_threshold=0)
plt.xlabel("Sequência")
plt.ylabel("Distância (Ward)")
plt.title("Dendrograma das sequências (descritor robusto)")
plt.tight_layout()
plt.savefig("figures/PartIV10_robusto.jpg", dpi=300)

for nome, ZZ in (("mediana", Z_med), ("robusto", Z_rob)):
    for K in (2, 3, 4):
        grupos = fcluster(ZZ, t=K, criterion="maxclust")
        print(f"\nDendrograma ({nome}) cortado em {K} grupos -> tamanhos:",
              np.bincount(grupos)[1:])


# -------------------------------------------------------------------
# Estabilidade: os grupos dependem do descritor? (ARI)
# -------------------------------------------------------------------
print("\nEstabilidade (ARI entre descritores):")
for K in (2, 3, 4):
    _, z_med = kmeans(F_norm, K, "km++")
    _, z_p90 = kmeans(F90_norm, K, "km++")
    _, z_rob = kmeans(F_rob_norm, K, "km++")
    print(f"  K = {K}: ARI(mediana, p90) = {adjusted_rand_score(z_med, z_p90):.3f}"
          f" | ARI(mediana, robusto) = {adjusted_rand_score(z_med, z_rob):.3f}"
          f" | ARI(p90, robusto) = {adjusted_rand_score(z_p90, z_rob):.3f}")
    print(f"        tamanhos: mediana {np.bincount(z_med)} | p90 {np.bincount(z_p90)}"
          f" | robusto {np.bincount(z_rob)}")


# -------------------------------------------------------------------
# Diagnósticos: o que está a separar os grupos?
# -------------------------------------------------------------------
t_med = np.array([np.median(tronco_seq(s)) for s in seqs])
t_min = np.array([tronco_seq(s).min() for s in seqs])
print("\nTronco mediano por sequência (min/mediana/max):",
      t_med.min().round(3), np.median(t_med).round(3), t_med.max().round(3))
print("Piores razões tronco mínimo / tronco mediano:", np.sort(t_min / t_med)[:5].round(2))

ext = np.abs(F90_norm).max(axis=1)
print("Sequências com valores mais extremos no p90:", np.argsort(ext)[::-1][:8])
print("  valores:", np.sort(ext)[::-1][:8].round(1))

for nome, Fn in (("p90", F90_norm), ("robusto", F_rob_norm)):
    _, z = kmeans(Fn, 2, "km++")
    for g in range(2):
        idx = np.where(z == g)[0]
        print(f"Grupo {nome} {g}: n = {len(idx)}, comprimento mediano = {np.median(comprimentos[idx])},"
              f" tronco mediano = {np.median(t_med[idx]):.3f}")

# Cruzamento com a Q9 (quando existir o agrupamento por exercício):
# print(pd.crosstab(grupo_exercicio, z_rob))

# k-means com vários arranques: fica com o de menor custo J
def kmeans_melhor(X, K, flag="km++", n=20):
    melhor = None
    for _ in range(n):
        C, z = kmeans(X, K, flag)
        J = kmeans_custo(X, C, z)[1]
        if melhor is None or J < melhor[0]:
            melhor = (J, C, z)
    return melhor[1], melhor[2]

# Estabilidade entre metades: agrupa os descritores calculados só com a 1.ª
# metade e só com a 2.ª metade de cada sequência (Ward, determinístico).
# Se os grupos refletirem algo estável de cada sequência, devem coincidir.
FA = StandardScaler().fit_transform(np.array([descritor_robusto(s[:len(s) // 2]) for s in seqs]))
FB = StandardScaler().fit_transform(np.array([descritor_robusto(s[len(s) // 2:]) for s in seqs]))
ZA, ZB = linkage(FA, method="ward"), linkage(FB, method="ward")

print("\nEstabilidade entre metades (ARI, Ward):")
for K in (2, 3, 4):
    gA = fcluster(ZA, t=K, criterion="maxclust")
    gB = fcluster(ZB, t=K, criterion="maxclust")
    g = fcluster(Z_rob, t=K, criterion="maxclust")
    print(f"  K = {K}: ARI(A, B) = {adjusted_rand_score(gA, gB):.3f}"
          f" | ARI(A, completo) = {adjusted_rand_score(gA, g):.3f}"
          f" | ARI(B, completo) = {adjusted_rand_score(gB, g):.3f}")

# Os 3 grupos do robusto estão a separar o comprimento das sequências?
g3 = fcluster(Z_rob, t=3, criterion="maxclust")
for k in np.unique(g3):
    idx = np.where(g3 == k)[0]
    print(f"Grupo {k}: n = {len(idx)}, comprimento mediano = {np.median(comprimentos[idx])},"
          f" intervalo = {comprimentos[idx].min()}-{comprimentos[idx].max()}")

# ARI robusto vs mediana com 20 arranques por K
print("\nARI com vários arranques:")
for K in (2, 3, 4):
    _, za = kmeans_melhor(F_rob_norm, K)
    _, zb = kmeans_melhor(F_norm, K)
    print(f"  K = {K}: ARI(robusto, mediana) = {adjusted_rand_score(za, zb):.3f}"
          f" | tamanhos: {np.bincount(za)} e {np.bincount(zb)}")

# 1) O descritor robusto depende do comprimento da sequência?
logL = np.log(comprimentos)
print("\nSpearman entre cada descritor robusto e log(comprimento):")
for j, (a, b) in enumerate(PARES):
    rho, _ = spearmanr(F_rob[:, j], logL)
    print(f"  segmento {a:2d}-{b:2d}: rho = {rho:+.2f}")

# 2) Descritor em janelas do mesmo tamanho (30 poses = comprimento mínimo)
def janela_central(seq, n=30):
    ini = (len(seq) - n) // 2
    return seq[ini:ini + n]

F_jan = StandardScaler().fit_transform(
    np.array([descritor_robusto(janela_central(s)) for s in seqs])
)
Z_jan = linkage(F_jan, method="ward")

print("\nJanelas de 30 poses (comprimento igual para todas):")
for K in (2, 3):
    g_jan = fcluster(Z_jan, t=K, criterion="maxclust")
    g = fcluster(Z_rob, t=K, criterion="maxclust")
    print(f"  K = {K}: ARI(janela, completo) = {adjusted_rand_score(g_jan, g):.3f}"
          f" | tamanhos: {np.bincount(g_jan)[1:]}")

# 3) Que sequência fica isolada em K = 4?
g4 = fcluster(Z_rob, t=4, criterion="maxclust")
print("\nSequências isoladas em K = 4:", np.where(np.bincount(g4)[g4] == 1)[0])

plt.show()   # última linha do ficheiro
